# <center>TensorRT 简介</center>

<center><img src='./course_data/images/tensorrt-logo.png'></center>

## <center>NVIDIA® TensorRT™</center>

[NVIDIA® TensorRT™](https://developer.nvidia.com/tensorrt) 是一个用于高性能深度学习推理的API生态系统。**TensorRT 包括推理运行时环境和模型优化器，为生产应用程序提供低延迟和高吞吐量。** TensorRT 生态系统包括 [TensorRT](https://github.com/NVIDIA/TensorRT), [TensorRT-LLM](https://github.com/NVIDIA/TensorRT-LLM), [TensorRT Model Optimizer](https://github.com/NVIDIA/TensorRT-Model-Optimizer) 和 [TensorRT Cloud](https://docs.nvidia.com/deeplearning/tensorrt-cloud/latest/index.html)。

### 什么是 TensorRT

- **用于高性能深度学习推理的 SDK**
- 内含深度学习推理运行时环境和模型优化器
- 可为推理应用程序**提供低延迟和高吞吐量**
- 有 [C++](https://docs.nvidia.com/deeplearning/tensorrt/api/c_api/index.html) 与 [Python](https://docs.nvidia.com/deeplearning/tensorrt/api/python_api/index.html) 的API, 可等价混用


<center>
    <img src="./course_data/images/dl-cycle.png" width=100%>
    <div style="font-size: smaller; color: gray;">使用 TensorRT 的典型深度学习开发周期
    </div>
</center>

## <center>TensorRT 工作细节</center>

<center>
    <img src="./course_data/images/trt-pipeline.png" width=90%>
    <div style="font-size: smaller; color: gray;">TensorRT 工作流程：TensorRT 模型优化器（中间）和 TensorRT 推理运行时（右侧）
    </div>
</center>

**TensorRT的使用主要分为两个阶段：构建和部署。** 在构建阶段，**TensorRT对深度学习模型进行优化，包括层融合、内核选择和精度校准等，以生成一个针对特定硬件平台优化的推理引擎。** 这个引擎是一个序列化的二进制文件，可以存储在磁盘上，也可以加载到内存中进行部署。

在部署阶段，这个优化后的引擎被加载到GPU上，用于处理输入数据，执行推理计算，并输出结果。这一过程不需要在部署硬件上安装和运行完整的深度学习框架，从而减少了资源消耗并提高了推理速度。通过这种方式，TensorRT能够显著提升深度学习模型在NVIDIA GPU上的推理性能，适用于各种实时AI应用和服务。


### 构建阶段（Model Optimizations）

<center>
    <img src="./course_data/images/tensorrt_build_phase.png" width=90%>
    <div style="font-size: smaller; color: gray;">
        TensorRT 对训练好的神经网络模型进行优化，生成一个准备就绪的运行时推理引擎
    </div>
</center>

#### 层间融合与张量融合

<center>
    <img src="./course_data/images/layer_and_tensor_fusion.png" width=90%>
    <div style="font-size: smaller; color: gray;">
     TensorRT 通过垂直和水平层融合以及层消除优化，简化了 GoogleNet Inception 模块图，减少了计算和内存开销
    </div>
</center>

在推理过程中，当深度学习框架执行此图时，它需要为每个层进行多次函数调用。由于每个操作都在 GPU 上执行，这意味着需要多次启动 CUDA 内核。内核计算通常相对于内核启动开销和每层读写张量数据的成本来说是非常快的。这导致了内存带宽瓶颈和可用 GPU 资源的利用不足。

**TensorRT 通过垂直和水平层融合以及层消除来解决这个问题。总体而言，减少了内核启动次数，并避免了在层与层之间进行内存读写，从而减少了推理延迟。**

#### 精度校准

大多数深度学习框架在进行训练时采用的是全32位精度（Full 32-bit precision，FP32）。在推理部署阶段，由于不需要反向传播，可以适当降低数据精度，比如使用半精度FP16甚至INT8。**使用较低精度可以导致模型尺寸更小、内存使用和延迟更低，以及更高的吞吐量。**

#### 优选 Kernel

在优化阶段，TensorRT 会从数百个专门的内核中进行选择，许多这些内核都是针对一系列参数和目标平台手工调整和优化的。例如，执行卷积运算就有几种不同的算法。TensorRT 会从内核库中挑选出针对目标 GPU、输入数据大小、过滤器大小、张量布局、批量大小以及其他参数提供最佳性能的实现。这样做**确保了部署的模型针对特定的部署平台以及正在部署的特定神经网络都进行了性能优化。**

#### 显存优化

**TensorRT 通过为每个张量在其使用期间分配内存，从而减少了内存占用并提高了内存的重用率，避免了快速高效执行的内存分配开销。**

#### 多流执行

这种优化方式允许 TensorRT 并行处理多个输入流，从而提高 GPU 资源的利用率和整体的推理吞吐量。在多流执行模式下，TensorRT 能够为每个输入流分配独立的 CUDA 流，这些流可以并行执行，使得多个推理请求可以同时在 GPU 上进行处理。

#### 导入 Plugin

<center>
    <img src="./course_data/images/trtplugin.png" width=40%>
    <div style="font-size: smaller; color: gray;">
     自定义层可以作为插件集成到TensorRT运行时中
    </div>
</center>

深度学习是一个快速发展的领域，新类型的层经常被引入。TensorRT提供了一个自定义层API，能够定义那些原生不支持的自定义层。这些自定义层使用C++定义，这样做可以轻松利用高度优化的CUDA库，如cuDNN和cuBLAS。

### 部署阶段 （Inference Runtime）

- **序列化与反序列化：** 保存和加载推理引擎，便于模型的分发和重用
- **资源管理：** 配置输入输出缓冲区，管理内存和显存使用
- **生命周期管理：** 创建和管理推理引擎的生命周期
- **异常处理：** 确保推理过程中的稳定性和错误管理
- **推理执行：** 创建执行上下文并执行推理任务

## <center>TensorRT 工作流程</center>

<center>
    <img src="./course_data/images/trt_workflow.png" width=35%>
    <div style="font-size: smaller; color: gray;">
     TensorRT 工作流程的五个基本步骤
    </div>
</center>

**在 TensorRT 的工作流程中，需要依次执行以下五个关键步骤：首先导出模型，接着确定合适的批量大小，然后选择所需的计算精度，之后将模型转换为 TensorRT 引擎，最后部署并运行模型。这些步骤构成了转换和部署模型的基础框架，而选择合适的转换和部署策略则确保模型能够在各种环境中达到最佳性能。**

### 转换和部署策略

<center>
    <img src="./course_data/images/conversion-runtime.png" width=55%>
</center>

| 方法 | 易用性 | 性能 | 兼容性 | 开发效率 | 不支持的算子 |
| ---- | ------ | ---- | ------ | -------- | ------------ |
| **框架自带接口 ( [Torch-TensorRT](https://pytorch.org/TensorRT/),[TF-TRT](https://docs.nvidia.com/deeplearning/frameworks/tf-trt-user-guide/index.html))** | ★★★ | ★ | ★★ | ★★★ | 使用框架计算 |
| **ONNX** | ★★ | ★★☆ | ★★☆ | ★★ | 修改网络/ONNX、 写Plugin |
| **TensorRT API手动构建** | ★ | ★★★ | ★★★ | ★ | 写Plugin |

## <center>总结</center>

本节课程中，我们详细介绍了 NVIDIA® TensorRT™ 的核心概念、工作流程以及优化策略。TensorRT 通过其强大的模型优化和推理运行时环境，能够显著提升深度学习模型的推理性能，适用于各种实时AI应用和服务。

在实际应用中，选择合适的转换和部署策略非常重要。无论是使用框架自带接口、ONNX，还是手动构建 TensorRT API，都需要根据具体的应用场景和需求进行权衡。

### 扩展阅读：

- [NVIDIA® TensorRT™ 快速入门指南](https://docs.nvidia.com/deeplearning/tensorrt/latest/getting-started/quick-start-guide.html#quick-start-guide)